<a href="https://colab.research.google.com/github/reyhane6/Data-science/blob/main/gpt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as f

In [5]:
@dataclass
class GPTConfig:
  vocab_size=50257
  block_size=1024
  n_head=12
  n_layer=12
  n_embd=768

In [6]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)

        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, X):
        B, T, C = X.size()
        qkv = self.attn(X)

        q, k, v = qkv.split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, C//self.head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C//self.head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.head).transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        y = self.c_proj(y)

        return y


In [7]:
class MLP(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.c_fc=nn.Linear(config.n_embd,4*config.n_embd)
    self.gelu=nn.GELU(approximate='tanh')
    self.c_proj=nn.Linear(4*config.n_embd,config.n_embd)
    self.gelu=nn.GELU()
  def forward(self,X):
    X=self.c_fc(X)
    X=self.gelu(X)
    X=self.c_proj(X)
    return X

In [8]:
class Block(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.ln1=nn.LayerNorm(config.n_embd)
    self.attn=CausalSelfAttention(config)
    self.ln2=nn.LayerNorm(config.n_embd)
    self.mlp=MLP(config)
  def forward(self,X):
    X=X+ self.attn(self.ln1(X))
    X=X+self.mlp(self.ln2(X))
    return X

In [9]:
class GPT2(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.config=config
    self.transformer=nn.ModuleDict(dict(
        wte=nn.Embedding(config.vocab_size,config.n_embd),
        wpe=nn.Embedding(config.block_size,config.n_embd),
        h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
        ln_f=nn.LayerNorm(config.n_embd)
    ))
    self.lm_head=nn.Linear(config.n_embd,config.vocab_size,bias=False)
  def forward(self,idx,tragets=None):
    B,T=idx.size()
    assert T<= self.config.block_size,f'Can not foreard sequence of length{T}'
    pos=torch.arange(0,T,dtype=torch.long,device=idx.device)
    pos_emb=self.transformer.wpe(pos)
    tok_emb=self.transformer.wte(idx)
    X=pos_emb+tok_emb
    for block in self.transformer.h:
      X=block(X)
      X=self.transformer.ln_f(x)
      logits=self.lm_head(X)
      loss=None
      if targets is not None:
        loss=F.cross_entropy(logits.view(-1,logits.size(-1)),targets.view(-1))
        return logits,loss

In [10]:
GPT2(GPTConfig)

GPT2(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)